# Practical RAG: Hybrid Search, Reranking, Citations & Faithfulness

This notebook extends the basic PDF RAG example with three practical improvements:

1. **Hybrid search** combines vector similarity with keyword search.
2. **Reranking** reorders retrieved passages so the most useful ones come first.
3. **Citations and faithfulness** show page-level sources and make it easier to verify the answer against the evidence.

Keep this notebook beside the same `sample_pdfs` folder used by the original notebook. Add your OpenAI key to a `.env` file. Run the cells from top to bottom.

## 1. Install packages and set up the API key

BM25 provides keyword search. The other packages load PDFs, create embeddings, and call the language model.

In [ ]:
%pip install -q langchain langchain-classic langchain-community langchain-openai langchain-chroma langchain-text-splitters chromadb pypdf python-dotenv rank_bm25

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Add OPENAI_API_KEY to a .env file or your environment, then rerun this cell.")

CHAT_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

## 2. Load and split the PDF documents

Each page keeps its filename and page number as metadata. We will use that metadata later to cite the source.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_dir = Path("sample_pdfs")
pdf_files = sorted(pdf_dir.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError("Put the sample_pdfs folder beside this notebook.")

pages = []
for pdf_file in pdf_files:
    pages.extend(PyPDFLoader(str(pdf_file)).load())

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(pages)
print(f"Loaded {len(pdf_files)} PDFs and created {len(chunks)} chunks.")

## 3. Create vector and keyword retrievers

Vector search finds passages with similar meaning. BM25 keyword search is good at matching exact terms such as policy names, amounts, and product names. Hybrid search combines both result lists.

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_hybrid_demo",
)
vector_retriever = vector_db.as_retriever(search_kwargs={"k": 5})

keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever.k = 5

hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, keyword_retriever],
    weights=[0.5, 0.5],
)

llm = ChatOpenAI(model=CHAT_MODEL, temperature=0)
print("Vector, keyword, and hybrid retrievers are ready.")

### Try hybrid search

Compare the results from vector search alone with the combined hybrid results. For a query containing a specific phrase or number, keyword search may help surface the exact passage.

In [ ]:
def show_results(retriever, question, limit=5):
    docs = retriever.invoke(question)
    for i, doc in enumerate(docs[:limit], start=1):
        source = Path(doc.metadata["source"]).name
        page = doc.metadata.get("page", "?")
        print(f"{i}. {source}, page {page + 1 if isinstance(page, int) else page}")
        print(doc.page_content[:350].replace("\n", " "))
        print()
    return docs

question = "What approval is needed for a single expense above 10,000 rupees?"
print("VECTOR SEARCH")
vector_docs = show_results(vector_retriever, question)
print("HYBRID SEARCH")
hybrid_docs = show_results(hybrid_retriever, question)

## 4. Rerank the retrieved passages

Hybrid search gathers candidates from two search methods. Reranking then orders those candidates by how well their text matches the question. This small teaching example uses word overlap so its behavior is easy to inspect. Production systems often use a trained reranker model for better semantic matching.

In [ ]:
import re

STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "what", "when", "how", "do", "does", "for", "to", "of", "in", "on", "and", "or", "be", "under"}

def terms(text):
    return {word for word in re.findall(r"[a-zA-Z0-9]+", text.lower()) if word not in STOP_WORDS}

def rerank(question, documents, limit=3):
    query_terms = terms(question)
    scored = []
    for doc in documents:
        doc_terms = terms(doc.page_content)
        score = len(query_terms & doc_terms) / max(len(query_terms), 1)
        scored.append((score, doc))
    return sorted(scored, key=lambda item: item[0], reverse=True)[:limit]

candidates = hybrid_retriever.invoke(question)
ranked_docs = rerank(question, candidates, limit=3)
for rank, (score, doc) in enumerate(ranked_docs, start=1):
    print(f"Rank {rank} | word-overlap score: {score:.2f} | {Path(doc.metadata['source']).name}")
    print(doc.page_content[:350].replace("\n", " "))
    print()

## 5. Answer with page-level citations

We pass the top reranked passages to the model and ask it to cite the provided source labels. PDF page numbers in the loader start at zero, so the code adds one for reader-friendly page numbering.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

ranked_context = []
for _, doc in ranked_docs:
    source = Path(doc.metadata["source"]).name
    page = doc.metadata.get("page", 0) + 1
    ranked_context.append(f"[Source: {source}, page {page}]\n{doc.page_content}")

context = "\n\n".join(ranked_context)
prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer using only the evidence in the context. If the answer is missing, say you could not find it. Cite supporting evidence with the exact source label, for example [Source: expense_policy.pdf, page 2]. Do not invent citations.

Context:
{context}"""),
    ("human", "{question}"),
])
answer = (prompt | llm).invoke({"context": context, "question": question})
print(answer.content)

## 6. Check faithfulness against the retrieved evidence

A citation helps a reader find the evidence; it does not guarantee that the answer is supported. Review each claim against the cited passage. This simple checklist is intentionally human-readable and avoids treating an automated score as proof.

In [ ]:
print("ANSWER TO VERIFY:")
print(answer.content)
print("\nRETRIEVED EVIDENCE:")
for _, doc in ranked_docs:
    source = Path(doc.metadata["source"]).name
    page = doc.metadata.get("page", 0) + 1
    print(f"\n[Source: {source}, page {page}]")
    print(doc.page_content)

print("\nReview: Are all factual claims supported by the cited passages? Do the cited pages contain the stated details?")

## Summary

- **Hybrid search** combines semantic similarity and exact keyword matching.
- **Reranking** moves the most relevant retrieved passages to the top before generation.
- **Citations and faithfulness checks** make answers easier to trace and verify.

The overlap-based reranker is a transparent demo. For a real application, evaluate a trained reranker and measure retrieval quality on representative questions.